# 01 · Cleaning — messy Titanic

**Dataset:** `data/titanic_raw.csv` — a deliberately dirty Titanic file.
**Covers:** load & first look · missing values · duplicates · dtypes · outliers.
**Time yourself:** ~35 minutes.

Work top to bottom. Say your reasoning out loud as you go — in the real interview the
*why* is worth more than the code. Solutions are in `solutions/01_cleaning_titanic.ipynb`.

In [375]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
df = pd.read_csv('data/titanic_raw.csv')
df.head()

,PassengerId,Survived,P Class,Name,Sex,Age,sib_sp,Parch,Ticket,Fare,Cabin,Embarked,TicketDate,CabinNote
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,$27.90,NaN,s,10/04/1912,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,$9.84,NaN,s,10/04/1912,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",FEMALE,NaN,0,2,2678,$15.25,NaN,C,1912-04-11,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",Female,29.0,1,0,2926,$26.00,NaN,S,1912-04-12,checked
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,$6.95,NaN,Q,"April 12, 1912",NaN


---

## Part A — First look

### Q1. How many rows and columns are there? Print the shape, the dtypes and the non-null counts in as few calls as you can.

<details><summary>hint</summary>

`.shape` and `.info()` — `.info()` gives you dtypes and non-null counts at once.

</details>

In [376]:
# your code here
print(df.shape, "\n")
print(df.dtypes, "\n")
print(df.isnull().sum())

(914, 14) 

PassengerId      int64
Survived         int64
P Class          int64
Name               str
Sex                str
Age            float64
sib_sp           int64
 Parch           int64
Ticket             str
Fare               str
Cabin              str
Embarked           str
TicketDate         str
CabinNote          str
dtype: object 

PassengerId      0
Survived         0
P Class          0
Name             0
Sex              0
Age            181
sib_sp           0
 Parch           0
Ticket           0
Fare             0
Cabin          704
Embarked         2
TicketDate       0
CabinNote      856
dtype: int64


In [377]:
df.info() # non-null counts and data types

<class 'pandas.DataFrame'>
RangeIndex: 914 entries, 0 to 913
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  914 non-null    int64  
 1   Survived     914 non-null    int64  
 2   P Class      914 non-null    int64  
 3   Name         914 non-null    str    
 4   Sex          914 non-null    str    
 5   Age          733 non-null    float64
 6   sib_sp       914 non-null    int64  
 7    Parch       914 non-null    int64  
 8   Ticket       914 non-null    str    
 9   Fare         914 non-null    str    
 10  Cabin        210 non-null    str    
 11  Embarked     912 non-null    str    
 12  TicketDate   914 non-null    str    
 13  CabinNote    58 non-null     str    
dtypes: float64(1), int64(5), str(8)
memory usage: 100.1 KB


### Q2. Get summary statistics for the numeric columns *and* for the object columns.
Looking at the output: name three columns whose dtype is **wrong** for what they
actually represent, and say what each should be.

<details><summary>hint</summary>

`df.describe()` defaults to numeric only. Pass `include='object'` for the rest.

</details>

In [378]:
# your code here
print(df.dtypes)
df.head()
# age, ticket, fare, ticket_date

PassengerId      int64
Survived         int64
P Class          int64
Name               str
Sex                str
Age            float64
sib_sp           int64
 Parch           int64
Ticket             str
Fare               str
Cabin              str
Embarked           str
TicketDate         str
CabinNote          str
dtype: object


,PassengerId,Survived,P Class,Name,Sex,Age,sib_sp,Parch,Ticket,Fare,Cabin,Embarked,TicketDate,CabinNote
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,$27.90,NaN,s,10/04/1912,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,$9.84,NaN,s,10/04/1912,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",FEMALE,NaN,0,2,2678,$15.25,NaN,C,1912-04-11,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",Female,29.0,1,0,2926,$26.00,NaN,S,1912-04-12,checked
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,$6.95,NaN,Q,"April 12, 1912",NaN


In [379]:
df.describe()

,PassengerId,Survived,P Class,Age,sib_sp,Parch
count,914.000000,914.000000,914.000000,733.000000,914.000000,914.000000
mean,443.814004,0.387309,2.307440,30.092319,0.518600,0.380744
std,258.993287,0.487402,0.836339,16.918047,1.097734,0.804901
min,1.000000,0.000000,1.000000,-5.000000,0.000000,0.000000
25%,219.250000,0.000000,2.000000,20.000000,0.000000,0.000000
50%,444.500000,0.000000,3.000000,28.000000,0.000000,0.000000
75%,667.750000,1.000000,3.000000,38.000000,1.000000,0.000000
max,891.000000,1.000000,3.000000,200.000000,8.000000,6.000000


In [380]:
df.describe(include="object")

C:\Users\Pablo\AppData\Local\Temp\ipykernel_2632\702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,Name,Sex,Ticket,Fare,Cabin,Embarked,TicketDate,CabinNote
count,914,914,914,914,210,912,914,58
unique,891,9,681,236,147,7,9,1
top,"Calderhead, Mr. Edward Pennington",male,1601,$8.05,G6,s,"April 11, 1912",checked
freq,2,131,8,44,4,233,110,58


### Q3. The column names are messy: `'P Class'`, `'sib_sp'`, `' Parch'`, `'PassengerId'`,
`'TicketDate'`. Normalise **all** of them to lowercase snake_case, without typing
each name out.

Watch out for the CamelCase ones — `TicketDate` should become `ticket_date`,
not `ticketdate`.

<details><summary>hint</summary>

`df.columns` is an Index with `.str` accessors, just like a Series. For the CamelCase split, insert an underscore at every lowercase→uppercase boundary with a zero-width regex: `r'(?<=[a-z0-9])(?=[A-Z])'`.

</details>

In [381]:
# your code here
df.columns = (df.columns
              .str.strip() # strip is used to remove leading and trailing whitespace from the column names
              .str.replace(r'(?<=[a-z0-9])(?=[A-Z])', '_', regex=True)  # CamelCase split
              .str.replace(' ', '_', regex=False)
              .str.lower())
print(df.columns.tolist())

['passenger_id', 'survived', 'p_class', 'name', 'sex', 'age', 'sib_sp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'ticket_date', 'cabin_note']


---

## Part B — Types & structure

### Q4. Convert `fare` to a proper numeric column. Watch out: values look like `$1,234.56`
and some are blank. Any value that cannot be parsed should become `NaN`, not raise.
How many NaNs did you end up with?

<details><summary>hint</summary>

Strip the `$` and the thousands separator first, then `pd.to_numeric(..., errors='coerce')`. `errors='coerce'` is what turns junk into NaN instead of raising.

</details>

In [382]:
df["fare"].isnull().sum()

np.int64(0)

In [383]:
df["fare"] = pd.to_numeric(df["fare"].str.replace("$", ""), errors="coerce")
df

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,cabin,embarked,ticket_date,cabin_note
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,NaN,s,10/04/1912,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,NaN,s,10/04/1912,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",FEMALE,NaN,0,2,2678,15.25,NaN,C,1912-04-11,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",Female,29.0,1,0,2926,26.00,NaN,S,1912-04-12,checked
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,6.95,NaN,Q,"April 12, 1912",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,580,1,3,"Jussila, Mr. Eiriik",male,32.0,0,0,STON/O 2. 3101286,7.92,NaN,S,10/04/1912,NaN
910,503,0,3,"O'Sullivan, Miss. Bridget Mary",FEMALE,NaN,0,0,330909,7.63,NaN,Q,12/04/1912,NaN
911,538,1,1,"LeRoy, Miss. Bertha",female,30.0,0,0,PC 17761,106.42,NaN,c,10/04/1912,NaN
912,197,0,3,"Mernagh, Mr. Robert",Male,NaN,0,0,368703,7.75,NaN,Q,"April 10, 1912",NaN


In [384]:
# or
#df["fare"] = df["fare"].str.split("$").str[1].astype(float) # is rigid. If it hits any value it cannot convert, it immediately crashes your code with a ValueError

### Q5. Normalise `sex` and `embarked`: strip whitespace and unify the casing so that
`sex` has exactly 2 unique values and `embarked` has exactly 3 (plus NaN).
Verify with `value_counts(dropna=False)`.

<details><summary>hint</summary>

`.str.strip()` then `.str.lower()` / `.str.upper()`. Always re-check with `value_counts(dropna=False)` — the `dropna=False` is what shows you the NaNs.

</details>

In [385]:
# your code here
df.sex.value_counts(dropna=False)

sex
male       131
 male      128
male       122
MALE       107
Male       102
female      91
FEMALE      86
female      83
Female      64
Name: count, dtype: int64

In [386]:
df["sex"] = df.sex.str.strip().str.lower()
df.sex.value_counts()

sex
male      590
female    324
Name: count, dtype: int64

In [387]:
df.embarked.value_counts()
df.embarked = df.embarked.str.strip().str.lower()
df.embarked.value_counts()

embarked
s    663
c    169
q     80
Name: count, dtype: int64

### Q6. `ticket_date` is stored as text in **three different formats** (`1912-04-10`,
`10/04/1912`, `April 10, 1912`). Parse it into a real datetime column so that
**zero** values fail to parse.

<details><summary>hint</summary>

`pd.to_datetime(..., format='mixed')` parses each value independently. You also need `dayfirst=True` so `10/04/1912` reads as 10 April, not 4 October.

</details>

In [388]:
df.ticket_date

0          10/04/1912
1          10/04/1912
2          1912-04-11
3          1912-04-12
4      April 12, 1912
            ...      
909        10/04/1912
910        12/04/1912
911        10/04/1912
912    April 10, 1912
913    April 11, 1912
Name: ticket_date, Length: 914, dtype: str

In [389]:
# your code here
df["ticket_date"] = pd.to_datetime(df.ticket_date, format="mixed")
df

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,cabin,embarked,ticket_date,cabin_note
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,NaN,s,1912-10-04,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,NaN,s,1912-10-04,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",female,NaN,0,2,2678,15.25,NaN,c,1912-04-11,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",female,29.0,1,0,2926,26.00,NaN,s,1912-04-12,checked
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,6.95,NaN,q,1912-04-12,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,580,1,3,"Jussila, Mr. Eiriik",male,32.0,0,0,STON/O 2. 3101286,7.92,NaN,s,1912-10-04,NaN
910,503,0,3,"O'Sullivan, Miss. Bridget Mary",female,NaN,0,0,330909,7.63,NaN,q,1912-12-04,NaN
911,538,1,1,"LeRoy, Miss. Bertha",female,30.0,0,0,PC 17761,106.42,NaN,c,1912-10-04,NaN
912,197,0,3,"Mernagh, Mr. Robert",male,NaN,0,0,368703,7.75,NaN,q,1912-04-10,NaN


### Q7. Cast `sex` and `embarked` to the `category` dtype. Before you do — check the
cardinality to justify it. How much memory did the DataFrame save?

<details><summary>hint</summary>

`.nunique()` for cardinality, `.astype('category')` to convert, `.memory_usage(deep=True)` — `deep=True` matters, otherwise strings aren't counted.

</details>

In [390]:
# your code here
print(df.sex.memory_usage(deep=True))
df.sex = df.sex.astype("category")
df.sex.memory_usage(deep=True)

49222


1154

In [391]:
print(df.embarked.memory_usage(deep=True))
df.embarked = df.embarked.astype("category")
df.embarked.memory_usage(deep=True)

45796


1196

---

## Part C — Duplicates

Do this **before** imputing: imputing first would compute statistics over rows that shouldn't be there.

### Q8. How many fully duplicated rows are there? How many rows share a `passenger_id` with another row? Are those two the same set of rows?

<details><summary>hint</summary>

`.duplicated()` with no args compares every column; `subset=['passenger_id']` compares only the key.

</details>

In [392]:
# your code here
print(df.duplicated().sum())
df.duplicated(subset="passenger_id").sum() # check for duplicates

15


np.int64(23)

### Q9. Drop the fully duplicated rows. Then inspect the rows that still share a
`passenger_id` — what is different about them, and what would you do?

<details><summary>hint</summary>

`keep=False` inside `.duplicated()` marks *every* member of a duplicate group, not just the repeats — that's how you see both sides to compare them.

</details>

In [393]:
# your code here
df.drop_duplicates(keep="first", inplace=True)
print(df.passenger_id.is_unique)
df[df.duplicated(subset="passenger_id", keep=False)].sort_values(by="passenger_id") # keep=False returns all duplicates, not just the first or last occurrence.

# They are identical except `age` differs by 1 year -- a conflicting record rather than
# a copy. passenger_id is supposed to be a unique key, so one of the two is wrong.
# Without a source of truth, keep one deterministically:
df = df.drop_duplicates(subset=['passenger_id'], keep='first')
df.passenger_id.is_unique

False


True

---

## Part D — Missing values

### Q10. Build a single table showing, per column, the count **and** the percentage of missing values — sorted worst-first, and only for columns that actually have any.

<details><summary>hint</summary>

`.isnull().sum()` gives counts, `.isnull().mean()` gives the fraction. Build a DataFrame from both and filter.

</details>

In [394]:
# your code here
pct_missing = df.isnull().sum() / len(df)
pct_missing[pct_missing > 1e-5].sort_values(ascending=False)

cabin_note    0.937149
cabin         0.771044
age           0.197531
embarked      0.002245
dtype: float64

### Q11. `cabin_note` is ~94% missing. Decide what to do with it and justify it in a comment.

<details><summary>hint</summary>

Above ~60% missing the usual call is to drop — unless the *missingness itself* is the signal, in which case you keep a binary indicator instead.

</details>

In [395]:
# your code here
df.cabin_note.value_counts(dropna=False)

cabin_note
NaN        835
checked     56
Name: count, dtype: int64

In [396]:
df["cabin_note"] = df.cabin_note.fillna("unchecked")
df.cabin_note = df.cabin_note.astype("category")
df.cabin_note.value_counts()

cabin_note
unchecked    835
checked       56
Name: count, dtype: int64

In [397]:
df.groupby("cabin_note").survived.mean() # it has no effect on survival rate, so we can drop it since it has high % of missing value

cabin_note
checked      0.339286
unchecked    0.386826
Name: survived, dtype: float64

In [398]:
df.drop(columns="cabin_note", inplace=True)

In [399]:
df.head()

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,cabin,embarked,ticket_date
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,NaN,s,1912-10-04
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,NaN,s,1912-10-04
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",female,NaN,0,2,2678,15.25,NaN,c,1912-04-11
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",female,29.0,1,0,2926,26.00,NaN,s,1912-04-12
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,6.95,NaN,q,1912-04-12


### Q12. `cabin` is ~77% missing — but here the missingness may itself be informative.
Instead of dropping it blindly, first **check**: is the survival rate different for
passengers with vs. without a recorded cabin? Then act on what you find.

<details><summary>hint</summary>

`.notna().astype(int)` builds the indicator. Then `groupby` it and take the mean of `survived` — the mean of a 0/1 column is the rate.

</details>

In [400]:
df.cabin.nunique() # 147 unique values, but 77% missing

147

In [401]:
# your code here
df["has_cabin"] = ~df.cabin.isnull()
df.groupby("has_cabin").survived.mean()

has_cabin
False    0.299854
True     0.666667
Name: survived, dtype: float64

In [402]:
df["deck"] = df.cabin.str[0]

df.groupby("deck").survived.mean()

deck
A    0.466667
B    0.744681
C    0.593220
D    0.757576
E    0.750000
F    0.615385
G    0.500000
T    0.000000
Name: survived, dtype: float64

In [403]:
df["deck"].unique()

<StringArray>
[nan, 'B', 'D', 'C', 'E', 'A', 'G', 'F', 'T']
Length: 9, dtype: str

In [404]:
# we can drop the cabin column since it has high % of missing value and high cardinality and keep the has_cabin column instead
df.drop(columns="cabin", inplace=True)
df["has_cabin"] = df.has_cabin.astype("int")
df.head()

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,embarked,ticket_date,has_cabin,deck
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,s,1912-10-04,0,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,s,1912-10-04,0,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",female,NaN,0,2,2678,15.25,c,1912-04-11,0,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",female,29.0,1,0,2926,26.00,s,1912-04-12,0,NaN
4,826,0,3,"Flynn, Mr. John",male,NaN,0,0,368323,6.95,q,1912-04-12,0,NaN


### Q13. `age` is ~20% missing. A global median is crude. Impute it with the median age
**of the passenger's own (`p_class`, `sex`) group** instead. Confirm zero NaNs
remain and that the overall distribution barely moved.

<details><summary>hint</summary>

`groupby(...)['age'].transform(...)` returns a Series aligned to the original index — that alignment is why `transform` works here and `apply`/`agg` wouldn't.

</details>

In [405]:
# your code here
#df["age"] = df.groupby(["p_class", "sex"])["age"].transform("median") # it won't work since it substitutes everywhere
df["age"] = df.groupby(["p_class", "sex"])["age"].transform(lambda x: x.fillna(x.median()))

### Q14. `embarked` has only a couple of missing values. Fill them with the mode.

<details><summary>hint</summary>

`.mode()` returns a Series (there can be ties) — take `[0]`.

</details>

In [406]:
# your code here
df.embarked.value_counts(dropna=False)
df.embarked = df.embarked.fillna(df.embarked.mode()[0])
df.head()

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,embarked,ticket_date,has_cabin,deck
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,s,1912-10-04,0,NaN
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,s,1912-10-04,0,NaN
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",female,21.5,0,2,2678,15.25,c,1912-04-11,0,NaN
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",female,29.0,1,0,2926,26.00,s,1912-04-12,0,NaN
4,826,0,3,"Flynn, Mr. John",male,25.0,0,0,368323,6.95,q,1912-04-12,0,NaN


### Q15. `deck` is still mostly missing. Fill it with an explicit `'Unknown'` category rather than a mode — why is that the better call here?

<details><summary>hint</summary>

Think about what mode-filling 77% of a column does to that column's distribution.

</details>

In [407]:
df["deck"].value_counts(dropna=False) # makes no sense to fillna with the mode, since it must be because they didnt have any cabine
# in addition, it would shift the distribution towards C, when C is no that far from other cabines

deck
NaN    687
C       59
B       47
D       33
E       32
A       15
F       13
G        4
T        1
Name: count, dtype: int64

In [408]:
# your code here
df["deck"] = df["deck"].fillna("unknown")
df["deck"] = df["deck"].astype("category")
df.head()

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,embarked,ticket_date,has_cabin,deck
0,64,0,3,"Skoog, Master. Harald",male,4.0,3,2,347088,27.90,s,1912-10-04,0,unknown
1,678,1,3,"Turja, Miss. Anna Sofia",female,18.0,0,0,4138,9.84,s,1912-10-04,0,unknown
2,141,0,3,"Boulos, Mrs. Joseph (Sultana)",female,21.5,0,2,2678,15.25,c,1912-04-11,0,unknown
3,54,1,2,"Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkin...",female,29.0,1,0,2926,26.00,s,1912-04-12,0,unknown
4,826,0,3,"Flynn, Mr. John",male,25.0,0,0,368323,6.95,q,1912-04-12,0,unknown


---

## Part E — Outliers

### Q16. Some `age` values are physically impossible. Find them. How many rows, and what
are the values?

<details><summary>hint</summary>

Ages below 0 or above ~100 aren't outliers, they're data-entry errors.

</details>

In [409]:
# your code here
df.age.min(), df.age.max()

(np.float64(-5.0), np.float64(200.0))

### Q17. Fix the impossible ages. Don't just delete the rows — the other columns in them are
fine. Treat the bad value as missing and re-impute it the same way as before.

<details><summary>hint</summary>

Dropping the row throws away 10 good columns to fix 1 bad cell. Set it to NaN and route it back through your imputation.

</details>

In [410]:
# your code here
df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = np.nan

df["age"] = df.groupby(["p_class", "sex"])["age"].transform(lambda x: x.fillna(x.median()))

### Q18. Count the `fare` outliers using the 1.5×IQR rule. Then look at the biggest ones —
are they errors, or real?

<details><summary>hint</summary>

`.quantile([0.25, 0.75])` returns both at once and unpacks into two variables.

</details>

In [411]:
# your code here
fare_q25, fare_q75 = df.fare.quantile([0.25, 0.75])
fare_iqr = fare_q75 - fare_q25
print(fare_iqr)
df[(df["fare"] < fare_q25 - 1.5*fare_iqr) | (df["fare"] > fare_q75 + 1.5*fare_iqr)][["fare", "has_cabin", "p_class"]].describe()

23.09


,fare,has_cabin,p_class
count,116.000000,116.000000,116.000000
mean,128.291810,0.775862,1.163793
std,84.637393,0.418823,0.509799
min,66.600000,0.000000,1.000000
25%,78.192500,1.000000,1.000000
50%,90.000000,1.000000,1.000000
75%,147.777500,1.000000,1.000000
max,512.330000,1.000000,3.000000


In [412]:
df[(df["fare"] < fare_q25 - 1.5*fare_iqr) | (df["fare"] > fare_q75 + 1.5*fare_iqr)]

,passenger_id,survived,p_class,name,sex,age,sib_sp,parch,ticket,fare,embarked,ticket_date,has_cabin,deck
7,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.00,s,1912-12-04,1,B
15,545,0,1,"Douglas, Mr. Walter Donald",male,50.0,1,0,PC 17761,106.42,c,1912-11-04,1,C
19,485,1,1,"Bishop, Mr. Dickinson H",male,25.0,1,0,11967,91.08,c,1912-04-11,1,B
25,588,1,1,"Frolicher-Stehli, Mr. Maxmillian",male,60.0,1,1,13567,79.20,c,1912-04-12,1,B
46,506,0,1,"Penasco y Castellana, Mr. Victor de Satode",male,18.0,1,0,PC 17758,108.90,c,1912-10-04,1,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
879,764,1,1,"Carter, Mrs. William Ernest (Lucile Polk)",female,36.0,1,2,113760,120.00,s,1912-04-12,1,B
884,326,1,1,"Young, Miss. Marie Grice",female,36.0,0,0,PC 17760,135.63,c,1912-12-04,1,C
898,731,1,1,"Allen, Miss. Elisabeth Walton",female,29.0,0,0,24160,211.34,s,1912-04-10,1,B
900,367,1,1,"Warren, Mrs. Frank Manley (Anna Sophia Atkinson)",female,60.0,1,0,110813,75.25,c,1912-04-11,1,D


### Q19. You decided the extreme fares are real. Handle them two ways and keep both columns:
(a) a winsorized `fare_capped` clipped at the 1st/99th percentile, and
(b) a `fare_log` using a log transform that tolerates the zero fares.
Compare the skew of all three.

<details><summary>hint</summary>

`.clip(lo, hi)` winsorizes. `np.log1p` = log(1+x), which is defined at x=0 — plain `np.log` would give you -inf on the free tickets.

</details>

In [413]:
# your code here
fare_q1, fare_q99 = df["fare"].quantile([0.01, 0.99])

# df.loc[df["fare"]<fare_q1, "fare"] = fare_q1
# df.loc[df["fare"]>fare_q99, "fare"] = fare_q99

df["fare_capped"] = df["fare"].clip(fare_q1, fare_q99)
df["fare_log"] = np.log1p(df["fare"]) # diff between log1p and log is that log1p computes log(1 + x) for all x, 
# which is more accurate for small x. It is also defined for x = -1, while log is not.

---

## Part F — Wrap up

### Q20. Final check, then save. Assert that: there are no missing values left, `passenger_id`
is unique, and `age`/`fare` are within sane ranges. Save to
`data/titanic_cleaned_by_me.csv`.

<details><summary>hint</summary>

`assert` statements are a nice thing to write in an interview — they show you think about verifying your own work.

</details>

In [417]:
# your code here
assert df.isnull().sum().sum() == 0
assert df.passenger_id.is_unique
assert df['age'].between(0, 100).all()
assert (df['fare'] >= 0).all()

df.to_csv('data/titanic_cleaned_by_me.csv', index=False)

---

## Debrief — what an interviewer is checking

- **Split before you fit.** Every imputation here was computed on the whole file. That is
  fine for an exploratory clean, but the moment a fill value comes from a *statistic*
  (median, mode, group mean), it must be learned on train only. Notebook 03 redoes this
  properly inside a `Pipeline`.
- **You checked before you dropped.** The `cabin` question is the trap: 77% missing looks
  like an easy drop, but the missingness carried most of the signal.
- **Errors vs. outliers.** A negative age is a bug; a £512 fare is a fact. Different fixes.
- **Order matters.** Duplicates before imputation; type fixes before anything numeric.